# Temporal Analysis: Trends and Cyclical Patterns

This notebook analyzes temporal patterns in Ulaanbaatar weather data.

## Objectives:
- Identify long-term temperature trends
- Detect cyclical patterns (yearly, multi-year)
- Perform time series decomposition
- Analyze seasonal trends
- Calculate temperature anomalies
- Detect extreme weather events

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append('../')

from config.config import *
from src.analysis.temporal_analysis import *
from src.visualization.plots import *

pd.set_option('display.max_columns', None)
%matplotlib inline

print("Libraries loaded!")

In [ ]:
# Load processed data
df_daily = pd.read_csv(PROCESSED_DATA_DIR / 'daily_aggregated.csv', index_col=0, parse_dates=True)
df_monthly = pd.read_csv(PROCESSED_DATA_DIR / 'monthly_aggregated.csv', index_col=0, parse_dates=True)
df_yearly = pd.read_csv(PROCESSED_DATA_DIR / 'yearly_aggregated.csv', index_col=0, parse_dates=True)
df_hourly = pd.read_csv(PROCESSED_DATA_DIR / 'hourly_processed.csv', index_col=0, parse_dates=True)

print(f"Loaded daily data: {len(df_daily)} rows")
print(f"Loaded monthly data: {len(df_monthly)} rows")
print(f"Loaded yearly data: {len(df_yearly)} rows")

## 1. Long-term Temperature Trend

In [ ]:
# Calculate linear trend
trend_result = calculate_trend(df_daily, column='temp_mean', method='linear')

print(f"\n{'='*60}")
print("TEMPERATURE TREND ANALYSIS")
print(f"{'='*60}")
print(f"Slope: {trend_result['slope']:.6f}°C per day")
print(f"Slope: {trend_result['slope'] * 365.25:.6f}°C per year")
print(f"Slope: {trend_result['slope'] * 365.25 * 10:.4f}°C per decade")
print(f"R²: {trend_result['r_squared']:.4f}")
print(f"P-value: {trend_result['p_value']:.2e}")
print(f"Total change (1990-2024): {trend_result['total_change']:.2f}°C")

In [ ]:
# Plot temperature with trend line
plot_temperature_trend(
    df_daily,
    column='temp_mean',
    title='Ulaanbaatar Temperature Trend (1990-2024)',
    trend_line=trend_result['trend_line'],
    save_path=FIGURES_DIR / '10_temperature_trend_with_line.png'
)
plt.show()

In [ ]:
# Mann-Kendall test for trend significance
mk_result = mann_kendall_test(df_daily, column='temp_mean')

## 2. Moving Averages

In [ ]:
# Calculate moving averages
df_daily_ma = calculate_moving_averages(
    df_daily,
    column='temp_mean',
    windows=[30, 365, 1825]  # 30 days, 1 year, 5 years
)

In [ ]:
# Plot moving averages
plot_moving_averages(
    df_daily_ma,
    column='temp_mean',
    windows=[365, 1825],
    title='Temperature with Moving Averages',
    save_path=FIGURES_DIR / '11_moving_averages.png'
)
plt.show()

## 3. Time Series Decomposition

In [ ]:
# Decompose time series
decomposition = decompose_time_series(
    df_daily,
    column='temp_mean',
    model='additive',
    period=365
)

In [ ]:
# Plot decomposition
plot_seasonal_decomposition(
    decomposition,
    title='Temperature Time Series Decomposition',
    save_path=FIGURES_DIR / '12_decomposition.png'
)
plt.show()

## 4. Cyclical Pattern Detection

In [ ]:
# Detect cyclical patterns using FFT
cyclical_result = detect_cyclical_patterns(
    df_daily,
    column='temp_mean',
    method='fft'
)

print("\nTop 10 Cyclical Periods:")
for i, period in enumerate(cyclical_result['periods'][:10], 1):
    print(f"{i}. Period: {period['period_days']:.1f} days ({period['period_days']/365.25:.2f} years)")
    print(f"   Power: {period['power']:.2e}")

## 5. Seasonal Trends

In [ ]:
# Analyze seasonal trends
seasonal_trends = analyze_seasonal_trends(df_hourly, temp_column='temp')

print("\nSeasonal Warming Rates:")
for season, trend in seasonal_trends.items():
    rate_per_decade = trend['slope'] * 365.25 * 10
    print(f"{season}: {rate_per_decade:.4f}°C per decade (p={trend['p_value']:.4f})")

In [ ]:
# Plot seasonal trends
plot_temperature_by_season(
    df_hourly,
    column='temp',
    save_path=FIGURES_DIR / '13_seasonal_trends.png'
)
plt.show()

## 6. Temperature Anomalies

In [ ]:
# Calculate temperature anomalies (baseline: 1990-2000)
df_yearly_anom = calculate_temperature_anomalies(
    df_yearly,
    column='temp_mean',
    baseline_period=(1990, 2000)
)

print("\nYearly Temperature Anomalies:")
print(df_yearly_anom[['year', 'temp_mean', 'temp_mean_anomaly']].tail(10))

In [ ]:
# Plot anomalies
plot_temperature_anomalies(
    df_yearly_anom,
    anomaly_column='temp_mean_anomaly',
    title='Temperature Anomalies (Baseline: 1990-2000)',
    save_path=FIGURES_DIR / '14_temperature_anomalies.png'
)
plt.show()

## 7. Extreme Events

In [ ]:
# Detect extreme temperature events
extreme_events = detect_extreme_events(
    df_daily,
    column='temp_mean',
    threshold_percentile=95
)

print(f"\nHottest days (>{extreme_events['high_threshold']:.2f}°C):")
print(extreme_events['hot_events'][['temp_mean']].head(10))

print(f"\nColdest days (<{extreme_events['low_threshold']:.2f}°C):")
print(extreme_events['cold_events'][['temp_mean']].head(10))

In [ ]:
# Analyze trend in extreme events
hot_events_yearly = extreme_events['hot_events'].groupby(
    extreme_events['hot_events'].index.year
).size()

cold_events_yearly = extreme_events['cold_events'].groupby(
    extreme_events['cold_events'].index.year
).size()

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(hot_events_yearly.index, hot_events_yearly.values, 
       alpha=0.7, color='red', label='Hot Events')
ax.bar(cold_events_yearly.index, -cold_events_yearly.values, 
       alpha=0.7, color='blue', label='Cold Events')
ax.axhline(y=0, color='black', linewidth=1)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of Events', fontsize=12)
ax.set_title('Extreme Temperature Events by Year', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '15_extreme_events.png', dpi=300)
plt.show()

## 8. Autocorrelation Analysis

In [ ]:
# Calculate autocorrelation
acf_result = calculate_autocorrelation(
    df_daily,
    column='temp_mean',
    nlags=365*2  # 2 years
)

In [ ]:
# Plot ACF and PACF
plot_autocorrelation(
    acf_result,
    title='Temperature Autocorrelation',
    save_path=FIGURES_DIR / '16_autocorrelation.png'
)
plt.show()

## Summary

This notebook completed:
- ✅ Long-term trend analysis (warming rate per decade)
- ✅ Moving average calculations
- ✅ Time series decomposition
- ✅ Cyclical pattern detection
- ✅ Seasonal trend analysis
- ✅ Temperature anomaly calculations
- ✅ Extreme event detection
- ✅ Autocorrelation analysis

**Key Findings:**
- Temperature warming rate: **[calculated above]**
- Strongest cyclical period: **[from FFT analysis]**
- Seasonal warming patterns: **[from seasonal trends]**

**Next Steps:**
- Proceed to `03_correlation_analysis.ipynb` for external factors analysis